In [ ]:
import os
import re
import csv
import cv2
import numpy as np
from google.colab import files

# =============================================================================
# CONFIGURACION
# =============================================================================

BASE_PROYECTO = "/content/proyecto_ia"

DIR_IMAGENES_ENTRENAMIENTO = os.path.join(BASE_PROYECTO, "imagenes_binarias_entrenamiento")
DIR_IMAGENES_PRUEBA        = os.path.join(BASE_PROYECTO, "imagenes_binarias_prueba")
DIR_CSV_ENTRENAMIENTO      = os.path.join(BASE_PROYECTO, "CSV_entrenamiento")
DIR_CSV_PRUEBA             = os.path.join(BASE_PROYECTO, "CSV_prueba")

RUTA_CSV_ENTRENAMIENTO = os.path.join(DIR_CSV_ENTRENAMIENTO, "datos_entrenamiento.csv")
RUTA_CSV_PRUEBA        = os.path.join(DIR_CSV_PRUEBA, "datos_prueba.csv")

ANCHO, ALTO = 128, 128
EXTENSIONES_VALIDAS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

# =============================================================================
# FUNCIONES
# =============================================================================

def crear_estructura_carpetas():
    for ruta in [DIR_IMAGENES_ENTRENAMIENTO, DIR_IMAGENES_PRUEBA,
                 DIR_CSV_ENTRENAMIENTO, DIR_CSV_PRUEBA]:
        if not os.path.exists(ruta):
            os.makedirs(ruta)
            print("Carpeta creada: " + ruta)

def subir_imagenes(destino):
    print("\nSelecciona las imagenes para: " + os.path.basename(destino))
    subidos = files.upload()
    for nombre in subidos.keys():
        os.rename(nombre, os.path.join(destino, nombre))
    print("Se han movido " + str(len(subidos)) + " archivos a " + destino)

def leer_imagen_binaria_como_vector(ruta_imagen):
    imagen = cv2.imread(ruta_imagen, cv2.IMREAD_GRAYSCALE)
    if imagen is None:
        raise ValueError("No se pudo leer: " + ruta_imagen)
    imagen = cv2.resize(imagen, (ANCHO, ALTO), interpolation=cv2.INTER_NEAREST)
    imagen_binaria = np.where(imagen > 127, 1, 0).astype(np.uint8)
    return imagen_binaria.flatten().tolist()

def obtener_imagenes_validas(directorio):
    return [os.path.join(directorio, f) for f in os.listdir(directorio)
            if os.path.isfile(os.path.join(directorio, f))
            and f.lower().endswith(EXTENSIONES_VALIDAS)]

def extraer_numero_entrenamiento(ruta_imagen):
    nombre = os.path.splitext(os.path.basename(ruta_imagen))[0]
    coincidencia = re.match(r"^[12]\.(\d+)$", nombre)
    return int(coincidencia.group(1)) if coincidencia else 999999

def extraer_numero_prueba(ruta_imagen):
    nombre = os.path.splitext(os.path.basename(ruta_imagen))[0].lower()
    coincidencia = re.search(r"(\d+)$", nombre)
    return int(coincidencia.group(1)) if coincidencia else 999999

def clasificar_imagen_entrenamiento(ruta_imagen):
    nombre = os.path.basename(ruta_imagen)
    if nombre.startswith("1."): return 0
    if nombre.startswith("2."): return 1
    raise ValueError("Nombre invalido en entrenamiento: " + nombre)

def clasificar_imagen_prueba(ruta_imagen):
    nombre = os.path.basename(ruta_imagen).lower()
    if nombre.startswith(("negativa", "negativo")): return 0
    if nombre.startswith(("positiva", "positivo")): return 1
    raise ValueError("Nombre invalido en prueba: " + nombre)

def generar_csv(directorio_imgs, ruta_csv, func_clasificar, func_extraer, titulo):
    separador = "=" * 40
    print("\n" + separador)
    print(titulo)
    print(separador)
    imgs = obtener_imagenes_validas(directorio_imgs)
    imgs.sort(key=func_extraer)

    if not imgs:
        print("Aviso: No hay imagenes en " + directorio_imgs)
        return

    with open(ruta_csv, mode="w", newline="") as f:
        escritor = csv.writer(f)
        # Encabezado: pixel_1, pixel_2, ..., pixel_16384, etiqueta
        n_pixeles = ANCHO * ALTO
        encabezado = ["pixel_" + str(i + 1) for i in range(n_pixeles)] + ["etiqueta"]
        escritor.writerow(encabezado)

        for i, ruta in enumerate(imgs, 1):
            etiqueta = func_clasificar(ruta)
            vector   = leer_imagen_binaria_como_vector(ruta)
            escritor.writerow(vector + [etiqueta])
            print("Procesado " + str(i).zfill(2) + ": " + os.path.basename(ruta) + " -> Label " + str(etiqueta))

    print("Archivo guardado en : " + ruta_csv)
    print("Total procesadas    : " + str(len(imgs)) + " imagenes")

# =============================================================================
# EJECUCION
# =============================================================================

crear_estructura_carpetas()

# Sube tus imagenes de entrenamiento (positivas: 2.x.png, negativas: 1.x.png)
subir_imagenes(DIR_IMAGENES_ENTRENAMIENTO)

# Sube tus imagenes de prueba (positivas: positiva_x.png, negativas: negativa_x.png)
subir_imagenes(DIR_IMAGENES_PRUEBA)

# Genera los CSVs
generar_csv(DIR_IMAGENES_ENTRENAMIENTO, RUTA_CSV_ENTRENAMIENTO,
            clasificar_imagen_entrenamiento, extraer_numero_entrenamiento, "ENTRENAMIENTO")

generar_csv(DIR_IMAGENES_PRUEBA, RUTA_CSV_PRUEBA,
            clasificar_imagen_prueba, extraer_numero_prueba, "PRUEBA")

print("\nProceso finalizado.")



Selecciona las imagenes para: imagenes_binarias_entrenamiento


Saving 1.1.png to 1.1.png
Saving 1.2.png to 1.2.png
Saving 1.3.png to 1.3.png
Saving 1.4.png to 1.4.png
Saving 1.5.png to 1.5.png
Saving 1.6.png to 1.6.png
Saving 1.7.png to 1.7.png
Saving 1.8.png to 1.8.png
Saving 1.9.png to 1.9.png
Saving 1.10.png to 1.10.png
Saving 1.11.png to 1.11.png
Saving 1.12.png to 1.12.png
Saving 1.13.png to 1.13.png
Saving 1.14.png to 1.14.png
Saving 1.15.png to 1.15.png
Saving 2.1.png to 2.1.png
Saving 2.2.png to 2.2.png
Saving 2.3.png to 2.3.png
Saving 2.4.png to 2.4.png
Saving 2.5.png to 2.5.png
Saving 2.6.png to 2.6.png
Saving 2.7.png to 2.7.png
Saving 2.8.png to 2.8.png
Saving 2.9.png to 2.9.png
Saving 2.10.png to 2.10.png
Saving 2.11.png to 2.11.png
Saving 2.12.png to 2.12.png
Saving 2.13.png to 2.13.png
Saving 2.14.png to 2.14.png
Saving 2.15.png to 2.15.png
Se han movido 30 archivos a /content/proyecto_ia/imagenes_binarias_entrenamiento

Selecciona las imagenes para: imagenes_binarias_prueba


Se han movido 0 archivos a /content/proyecto_ia/imagenes_binarias_prueba

ENTRENAMIENTO
Procesado 01: 1.1.png -> Label 0
Procesado 02: 2.1.png -> Label 1
Procesado 03: 2.2.png -> Label 1
Procesado 04: 1.2.png -> Label 0
Procesado 05: 1.3.png -> Label 0
Procesado 06: 2.3.png -> Label 1
Procesado 07: 2.4.png -> Label 1
Procesado 08: 1.4.png -> Label 0
Procesado 09: 2.5.png -> Label 1
Procesado 10: 1.5.png -> Label 0
Procesado 11: 1.6.png -> Label 0
Procesado 12: 2.6.png -> Label 1
Procesado 13: 1.7.png -> Label 0
Procesado 14: 2.7.png -> Label 1
Procesado 15: 1.8.png -> Label 0
Procesado 16: 2.8.png -> Label 1
Procesado 17: 1.9.png -> Label 0
Procesado 18: 2.9.png -> Label 1
Procesado 19: 1.10.png -> Label 0
Procesado 20: 2.10.png -> Label 1
Procesado 21: 2.11.png -> Label 1
Procesado 22: 1.11.png -> Label 0
Procesado 23: 1.12.png -> Label 0
Procesado 24: 2.12.png -> Label 1
Procesado 25: 1.13.png -> Label 0
Procesado 26: 2.13.png -> Label 1
Procesado 27: 2.14.png -> Label 1
Procesado 28